In [ ]:
import scvi
import scanpy as sc

In [ ]:
adata = sc.read_h5ad(
    "/workspace/data/251117_genomescale_CRISPRi/sample_mix_umi200_hvg500_pc25_neighbors10_mindist0.55.processed.h5ad"
)
adata

In [ ]:
adata_ = adata.copy()
adata_.X = adata_.layers["reads"]
sc.pp.filter_genes(adata_, min_cells=500)
sc.pp.highly_variable_genes(adata_, n_top_genes=500, flavor="seurat_v3")

In [ ]:
adata_ = adata_[:, adata_.var["highly_variable"]].copy()
adata_

In [ ]:
scvi.model.SCVI.setup_anndata(adata_, batch_key="batch")
model = scvi.model.SCVI(adata_, n_latent=25)
model.train(
    max_epochs=2000,
    batch_size=8192,
    early_stopping=True,
    early_stopping_monitor="elbo_validation",
    check_val_every_n_epoch=1,
    early_stopping_patience=25,
)

In [ ]:
model.history.keys()

In [ ]:
model.history["elbo_validation"].plot()

In [ ]:
latent = model.get_latent_representation()
adata.obsm["X_scVI"] = latent
sc.pp.neighbors(adata, use_rep="X_scVI")
sc.tl.umap(adata)
sc.pl.umap(adata)